# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution: Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL and contains clinical, pathological, and biomarker variables for 77 cancer survivors with second primary colorectal cancer.

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'
# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, their fields, columns, and their `@id`s.

We inspect the dataset's structure to see what record sets and fields are available for exploration. All objects are referenced using their Croissant `@id`.

In [ ]:
# List all record sets in the dataset by their @id and name
print('Available Record Sets:')
record_sets = list(dataset.record_sets)
for rs in record_sets:
    rs_id = rs['@id']
    name = rs.get('name', '')
    print(f"- @id: {rs_id}\tname: {name}")
    # List fields for each record set
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    for field in fields:
        field_id = field['@id'] if isinstance(field, dict) else field
        print(f"    - field @id: {field_id}")

### Example Record: Display a Sample Entry
Load and print the first record from each record set for inspection, using the record set `@id`.

In [ ]:
# For each record set, load and preview the first record
for rs in record_sets:
    rs_id = rs['@id']
    print(f"Sample from record set {rs_id}:")
    try:
        it = dataset.records(record_set=rs_id)
        first = next(it)
        print(first)
    except StopIteration:
        print("  (No records found)")
    except Exception as ex:
        print(f"  (Error: {ex})")

## 3. Data Extraction
Load data from the main record set(s) into pandas DataFrames for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Extract data from all record sets
dataframes = {}
for rs in record_sets:
    rs_id = rs['@id']
    try:
        # Read all records into DataFrame
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        print(f"Loaded record set {rs_id} with {len(df)} records, columns: {df.columns.tolist()}")
        dataframes[rs_id] = df
    except Exception as ex:
        print(f"Could not load records for {rs_id}: {ex}")

### Preview Main Clinical Data Table
For subsequent analysis, we'll identify the record set with the main clinical/pathological tabular data (likely the one containing columns such as 'Age', 'Sex', 'MSI_H_status', 'Comorbidity', etc.).

In [ ]:
# Heuristically identify the main data table: pick the largest record set loaded
main_rs_id = max(dataframes, key=lambda x: dataframes[x].shape[1] if x in dataframes else 0)
df = dataframes[main_rs_id]
print(f"Main record set ID: {main_rs_id}")
print("Columns:", df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)

We'll conduct basic EDA using the clinical record set. This includes filtering, normalization, and grouping using field `@id`s (i.e., column names corresponding to the schema).

In [ ]:
# Select a numeric field for filtering/normalizing, e.g., 'Age' if present.
numeric_field_id = None
possible_numeric_fields = ['Age', 'cr:Age', 'age', 'cr:age', 'cr:PatientAge', 'schema:age']
for f in possible_numeric_fields:
    if f in df.columns:
        numeric_field_id = f
        break
if numeric_field_id is None:
    raise ValueError('Could not identify a numeric field such as Age.')

# Cast to numeric, handle missing/coercion
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

threshold = 60
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold} (n={len(filtered_df)}):")
print(filtered_df.head())

# Normalize the numeric field
filtered_df = filtered_df.copy()
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print()
print(f"Normalized '{numeric_field_id}' for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Example grouping: by a categorical field such as sex or anatomical site
group_field = None
possible_group_fields = [
    'Sex', 'cr:Sex', 'sex', 'cr:sex', 'AnatomicalLocation', 'cr:AnatomicalLocation',
    'MSI_Status', 'cr:MSI_H_status', 'comorbidity', 'cr:Comorbidity'
]
for f in possible_group_fields:
    if f in df.columns or f in filtered_df.columns:
        group_field = f
        break

if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
    print()  # Line break
    print(f"Mean {numeric_field_id} by '{group_field}':")
    print(grouped_df.head())
else:
    print('(No suitable group field found for grouping analysis)')

## 5. Visualization

Plot distributions of the numeric field (Age) and optionally by grouping variable (e.g., by MSI status, sex, or anatomical site), using matplotlib or seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of age
plt.figure(figsize=(8,5))
sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# Compare age by group if available
if group_field:
    plt.figure(figsize=(8,5))
    sns.boxplot(x=group_field, y=numeric_field_id, data=df, showfliers=False)
    plt.title(f"{numeric_field_id} by {group_field}")
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()

## 6. Conclusion

- The FAIR² dataset provides structured clinical variables on second primary colorectal cancer in survivors, including demographic, pathological, and molecular (MSI/MMR) status information.

- Data exploration shows distributions and inter-variable relationships (e.g., age distribution, stratified by sex or molecular features).

- `mlcroissant` allows programmatic exploration and field-level referencing using Croissant `@id`s, supporting reproducible FAIR analysis workflows.

Further analysis (e.g., risk modeling, survival analysis) can build on these steps depending on study needs.